
# Synthetic Dataset Generator

A Gradio-powered notebook for generating structured AI training datasets using either OpenAI's GPT-4o-mini or a locally quantized Llama 3.2b model.

UI interface implemented in gradio, a user can define their business scenario in a dropdown or in details. 

User can also set number of records needed in the dataset. 

User can set format for export of either csv or json

User can set max output token to be used by the LLM




In [ ]:
!pip install -q gradio openai instructor pydantic pandas
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6


In [ ]:
# imports

import json
import re
import os
import pandas as pd
import gradio as gr
import torch

from typing import List
from pydantic import BaseModel

from openai import OpenAI
import instructor

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
# LLAMA MODEL CONFIGURATION (OPEN SOURCE MODEL)

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

try:
    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LLAMA,
        device_map="auto",
        quantization_config=quant_config
    )
    llama_available = True
except Exception as e:
    print(f"Warning: Could not load Llama model: {e}")
    llama_available = False


In [ ]:
#OPENAI MODEL SETUP
try:
    from google.colab import userdata
    openai_api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    openai_api_key = os.environ.get("OPENAI_API_KEY", "")

# FIX 1: instructor.patch() is deprecated — use instructor.from_openai()
try:
    client = instructor.from_openai(OpenAI(api_key=openai_api_key))
    openai_available = True
except Exception as e:
    print(f"Warning: Could not initialise OpenAI client: {e}")
    openai_available = False

In [ ]:

#DATASET SETUP

class DatasetRecord(BaseModel):
    instruction: str
    input: str
    output: str


class Dataset(BaseModel):
    records: List[DatasetRecord]


SCENARIOS = {
    "Customer support chatbot":
        "Generate conversations between customers and support agents resolving issues.",
    "HR interview questions":
        "Generate HR interview questions with strong model answers.",
    "Medical FAQ dataset":
        "Generate patient-friendly medical questions and answers.",
    "Programming help assistant":
        "Generate programming questions with explanations and solutions."
}

### Openai Frontier Model Dataset Generator

In [ ]:
# Openai Frontier Model Dataset Generator

def generate_openai_dataset(scenario, num_records, max_tokens):
    if not openai_available:
        raise RuntimeError("OpenAI client is not available. Check your API key.")

    prompt = f"""
Generate {num_records} synthetic dataset records for this scenario:

{scenario}

Each record must include:

instruction
input
output
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_model=Dataset,
        messages=[
            {"role": "system", "content": "You generate structured datasets for AI training."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens
    )

    return response.records


### LLAMA Open Source Model Dataset Generator

In [ ]:

# LLAMA Open Source Model Dataset Generator


def generate_llama_dataset(scenario, num_records, max_tokens):
    if not llama_available:
        raise RuntimeError("Llama model is not available. Check your GPU and model access.")

    prompt = f"""
    Generate {num_records} synthetic dataset examples for:

    {scenario}

    Format output as JSON list with fields:

    instruction
    input
    output
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text


In [ ]:
def save_dataset(records, file_format):
    data = [r.model_dump() for r in records]

    df = pd.DataFrame(data)
    path = f"/content/synthetic_dataset.{file_format}"

    if file_format == "csv":
        df.to_csv(path, index=False)
    elif file_format == "json":
        df.to_json(path, orient="records", indent=2)

    return path, df

### Generate dataset

In [ ]:
#generate dataset
def generate_dataset(model_choice,
                     scenario_choice,
                     custom_scenario,
                     num_records,
                     max_tokens,
                     file_format):
    try:
        scenario = custom_scenario.strip() if custom_scenario and custom_scenario.strip() else SCENARIOS[scenario_choice]

        if model_choice == "Frontier (OpenAI)":
            records = generate_openai_dataset(scenario, num_records, max_tokens)
            path, df = save_dataset(records, file_format)
            preview = df.head()

        else:
            text = generate_llama_dataset(scenario, num_records, max_tokens)

            json_match = re.search(r'\[.*\]', text, re.DOTALL)
            if not json_match:
                raise ValueError(
                    "Llama did not return valid JSON. Raw output:\n" + text[:500]
                )

            data = json.loads(json_match.group())
            records = [DatasetRecord(**r) for r in data]
            path, df = save_dataset(records, file_format)
            preview = df.head()

        return preview, path

    except Exception as e:
        error_df = pd.DataFrame([{"error": str(e)}])
        return error_df, None

## Lunch Gradio Interface

In [ ]:

# Gradio Interface


with gr.Blocks() as demo:

    gr.Markdown("# Synthetic Dataset Generator")

    model_choice = gr.Radio(
        ["Frontier (OpenAI)", "Open Source (Llama)"],
        label="Select Model"
    )

    scenario_choice = gr.Dropdown(
        list(SCENARIOS.keys()),
        label="Business Scenario"
    )

    custom_scenario = gr.Textbox(
        label="Custom Scenario (optional)"
    )

    num_records = gr.Slider(
        5, 200,
        value=20,
        label="Number of Records"
    )

    max_tokens = gr.Slider(
        100, 2000,
        value=800,
        label="Max Tokens"
    )

    file_format = gr.Radio(
        ["csv", "json", "parquet"],
        label="Output Format"
    )

    generate_btn = gr.Button("Generate Dataset")

    # FIX 5: preview must always be a DataFrame — Llama branch now also returns one
    preview = gr.Dataframe(label="Dataset Preview")

    download_file = gr.File(label="Download Dataset")

    generate_btn.click(
        generate_dataset,
        inputs=[
            model_choice,
            scenario_choice,
            custom_scenario,
            num_records,
            max_tokens,
            file_format
        ],
        outputs=[preview, download_file]
    )

demo.launch()